# Generate Images

In [ ]:
import os
import pandas as pd
import random

import tomllib

import avoddiag as ag

## Load Configuration

In [ ]:
with open("config/config.toml", "rb") as f:
    config = tomllib.load(f)

## Define Image Attributes

In [ ]:
# Base attribute definitions
attributes_def_base = pd.DataFrame(
    [
        dict(
            name='scene_type',
            type=str,
            options=[
                'dense',
                'desert',
                'forest',
                'industrial',
                'plain',
                'residential',
                'rural',
                'scrubland',
                'urban'
            ],
            vqa_query='Describe the scene type of this aerial view image using a single expression.'
        ),
        dict(
            name='season',
            type=str,
            options=['spring', 'summer', 'fall', 'winter'],
            vqa_query='Which season does this aerial view image represent? Use a single word.'
        ),
        dict(
            name='weather',
            type=str,
            options=[
                'cloudy',
                'dry',
                'snowy',
                'sunny'
            ],
            vqa_query='Describe the weather conditions in this aerial view image using a single expression.'
        ),
        dict(
            name='vehicle_count',
            type=int,
            options=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
            vqa_query='How many vehicles are there in the image? Answer with a number.'
        ),
        dict(
            name='vehicle_colors',
            type=list[str],
            options=['brown', 'camo', 'green', 'black', 'red', 'white', 'blue', 'grey'],
            vqa_query='Provide a comma-separated list of the colors of the vehicles in this aerial view image. Use "camo" if a vehicle is camouflaged.'
        )

    ]
)

attributes_def_base

In [ ]:
# Generate image attributes from the base attribute definitions - uniformly and independently
k = 100
random.seed(0)

dataset_attributes = {}
for i_attr, attr in attributes_def_base.iterrows():
    if attr['type'] in {str, int, float, bool}:
        values = random.choices(attr['options'], k=k)
        
    elif attr['type'] == list[str]:
        continue
        # values = [random.sample(attr['options'], k=random.randint(1, len(attr['options']))) for _ in range(k)]

    else:
        raise NotImplementedError(f"Attribute type {attr['type']} is not supported.")

    dataset_attributes[f"{attr['name']}"] = values

# vehicle_colors depend on vehicle_count, hence, it is generated after vehicle_count is available
vehicle_colors = []
for i, vehicle_count in enumerate(dataset_attributes['vehicle_count']):
    if vehicle_count == 0:
        vehicle_colors.append([])

    else:
        colors = sorted(
            set(
                random.choices(
                    attributes_def_base[attributes_def_base['name'] == 'vehicle_colors']['options'].values[0],
                    k=vehicle_count
                )
            )
        )
        vehicle_colors.append(colors)

dataset_attributes['vehicle_colors'] = vehicle_colors

dataset_attributes = pd.DataFrame(dataset_attributes)
dataset_attributes


## Compose Prompts for Image Generation

### Initialize OpenAI Provider

In [ ]:
provider_openai = ag.providers.openai.OpenAI(api_key=config['providers']['openai']['api_key'])

In [ ]:
# List models
models_openai = provider_openai.list_models()
# models_openai
models_openai[models_openai['id'].apply(lambda x: x.find('gpt-5') != -1)]  # Filter models that contain 'gpt-5' in their name

### Query Provider

In [ ]:
# Generate Prompts using OpenAI
from textwrap import dedent

instructions = dedent("""
    You are an expert prompt engineer generating realistic aerial image-generation prompts.

    You will receive a JSON array of records.

    Each record contains:
    - row_index (must be echoed unchanged)
    - scene_type
    - weather
    - season
    - vehicle_count
    - vehicle_colors
    - and possibly additional attributes.

    Generate exactly one output object for every input record.

    Requirements:

    1. Preserve every row.
    - Do not omit rows.
    - Do not merge rows.
    - Do not create extra rows.

    2. Echo row_index exactly as provided.

    3. Use only the provided attributes.
    - Do not invent attribute values.
    - If an attribute is missing or empty, make only the smallest realistic assumption.

    4. Prompt requirements:
    - straight-down (nadir, ~90° overhead) aerial view
    - photorealistic
    - neutral daylight unless weather or season implies otherwise
    - realistic scale
    - no artistic language
    - no camera brands or photography jargon
    - scene_type, weather, season, and vehicle attributes must be mutually consistent

    5. Vehicle colors:
    - If specified, include them.
    - If missing, use common realistic colors.

    6. Keep prompts concise:
    - one or two sentences
    - approximately 25–60 words

    Return only an object matching the PromptBatch schema.
""").strip()

dataset_prompts_data = provider_openai.generate_prompts(
    model_name="gpt-5",
    instructions=instructions,
    prompt_attributes=dataset_attributes,
)
dataset_prompts_data

In [ ]:
# Store prompts data (mainly for testing, datasets metadata will hold prompts)
os.makedirs("output", exist_ok=True)
dataset_prompts_data.to_json("output/dataset_prompts_data.json")

In [ ]:
# Load prompts data
# dataset_prompts_data = pd.read_json("output/dataset_prompts_data.json")
# dataset_prompts_data

## Generate Images

### Initialize Google Provider

In [ ]:
provider_google = ag.providers.google.GoogleGenAI(api_key=config['providers']['google']['api_key'])

In [ ]:
# List models
models_google = provider_google.list_models()
# models_google # Uncomment to show all Google models
# models_google[models_google.name.apply(lambda x: x.lower().find('imagen') != -1)] # Uncommend to show only Imagen models
# models_google[models_google.name.apply(lambda x: x.find('gemini-2.5') != -1)]  # Uncommend to show only Gemini 2.5 models
# models_google[models_google.name.apply(lambda x: x.find('gemini-3') != -1)]  # Uncommend to show only Gemini 3 models
# models_google[models_google.name.apply(lambda x: x.find('gemini-3.1') != -1)]  # Uncommend to show only Gemini 3.1 models
models_google[models_google.name.apply(lambda x: x.find('image') != -1)] 

### Query Provider

In [ ]:
# Create an empty image dataset
image_dataset = ag.image.data.Dataset(root_folder_path="output/TestDataset_01_Generated_Images")

In [ ]:
len(image_dataset)

In [ ]:
# Create an image generator object
# model_name = "imagen-3.0-generate-002" # DEPRICATED
# model_name = "imagen-4.0-generate-001" # DEPRICATED
model_name = "gemini-3-pro-image"
# model_name = "gemini-3.1-flash-image"

image_generator = ag.image.generation.Generator(provider=provider_google, model_name=model_name)


In [ ]:
image_generator.generate_dataset(
    image_dataset,
    metadata=dataset_prompts_data.copy()
)

In [ ]:
len(image_dataset)

In [ ]:
image_dataset.metadata['attributes_input']

In [ ]:
image_dataset[0]

In [ ]:
# Retrieve the image by file name
image_dataset['000001.png']['image']

In [ ]:
# Retrieve the image by index
import random

image_data = image_dataset[random.randint(0, len(image_dataset) - 1)]
print(image_data['metadata']['attributes_input']['prompt'])
image_data['image']
